# Day 6 — The four contexts

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** [The Four Contexts Where Every Polars Expression Lives](https://medium.com/python-in-plain-english/day-6-the-four-contexts-where-every-polars-expression-lives-42109f0d07ba)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Almost everything in Polars happens inside one of four **contexts**. An expression like `pl.col("price") * 2` does nothing on its own; the context decides what happens to its result.

| Context | Question it answers | Output |
|---|---|---|
| `select` | Which columns do I want? | only the columns you list |
| `with_columns` | What should I add or replace? | all columns + the new ones |
| `filter` | Which rows do I keep? | same columns, fewer rows |
| `group_by().agg()` | What is the summary per group? | one row per group |

> **Dates:** `.dt.total_days()` and `.dt.truncate()` below use Polars' own date types (`Datetime`, `Date`, `Duration`), not Python's `datetime` or pandas `Timestamp`. If they do not make sense yet, skip them; dates get their own posts later.

In [2]:
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
orders.head(3)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""e481f51cbdc54678b7cc49136f2d6a…","""9ef432eb6251297304e76186b10a92…","""delivered""",2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
"""53cdb2fc8bc7dce0b6741e21502734…","""b0830fb4747a6c6d20dea0b8c802d7…","""delivered""",2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
"""47770eb9100c2d0c44946d9cf07ec6…","""41ce2a54c0b03bf3443c3d931a3670…","""delivered""",2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


## 1. `select`: shape the output

In [3]:
orders.select(
    "order_id",
    "order_status",
    delivery_days=(pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")).dt.total_days(),
).head()

order_id,order_status,delivery_days
str,str,i64
"""e481f51cbdc54678b7cc49136f2d6a…","""delivered""",8
"""53cdb2fc8bc7dce0b6741e21502734…","""delivered""",13
"""47770eb9100c2d0c44946d9cf07ec6…","""delivered""",9
"""949d5b44dbf5de918fe9c16f97b45f…","""delivered""",13
"""ad21c59c0840e6cb83a9ceb5573f81…","""delivered""",2


In [4]:
# Aggregations in select collapse to one row
orders.select(
    n_orders=pl.len(),
    first_purchase=pl.col("order_purchase_timestamp").min(),
    last_purchase=pl.col("order_purchase_timestamp").max(),
)

n_orders,first_purchase,last_purchase
u32,datetime[μs],datetime[μs]
99441,2016-09-04 21:15:19,2018-10-17 17:30:18


## 2. `with_columns`: add or replace, keep everything

In [5]:
orders = orders.with_columns(
    delivery_days=(pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")).dt.total_days(),
    is_late=pl.col("order_delivered_customer_date") > pl.col("order_estimated_delivery_date"),
)
orders.select("order_id", "delivery_days", "is_late").head()

order_id,delivery_days,is_late
str,i64,bool
"""e481f51cbdc54678b7cc49136f2d6a…",8,false
"""53cdb2fc8bc7dce0b6741e21502734…",13,false
"""47770eb9100c2d0c44946d9cf07ec6…",9,false
"""949d5b44dbf5de918fe9c16f97b45f…",13,false
"""ad21c59c0840e6cb83a9ceb5573f81…",2,false


**Gotcha:** expressions in one `with_columns` call run **in parallel**, so they cannot see each other's results.

In [6]:
fresh = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
try:
    fresh.with_columns(
        delivery_days=(pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")).dt.total_days(),
        delivery_weeks=pl.col("delivery_days") / 7,  # not created yet!
    )
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

ColumnNotFoundError -> unable to find column "delivery_days"; valid columns: ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]


In [7]:
# Fix: chain two calls
fresh.with_columns(
    delivery_days=(pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")).dt.total_days()
).with_columns(delivery_weeks=pl.col("delivery_days") / 7).select("delivery_days", "delivery_weeks").head(3)

delivery_days,delivery_weeks
i64,f64
8,1.142857
13,1.857143
9,1.285714


## 3. `filter`: keep rows

In [8]:
late_delivered = orders.filter(
    pl.col("order_status") == "delivered",
    pl.col("is_late"),  # a boolean column is already a condition: keep rows where it is true
    # several conditions separated by commas = AND
)
print(f"{late_delivered.height:,} late deliveries out of {orders.height:,} orders")

7,826 late deliveries out of 99,441 orders


## 4. `group_by().agg()`: one row per group

In [9]:
(
    orders.group_by("order_status")
    .agg(
        n_orders=pl.len(),
        avg_delivery_days=pl.col("delivery_days").mean(),
        late_share=pl.col("is_late").mean(),
    )
    .sort("n_orders", descending=True)
)

order_status,n_orders,avg_delivery_days,late_share
str,u32,f64,f64
"""delivered""",96478,12.093604,0.081124
"""shipped""",1107,null,null
"""canceled""",625,19.833333,0.166667
"""unavailable""",609,null,null
"""invoiced""",314,null,null
"""processing""",301,null,null
"""created""",5,null,null
"""approved""",2,null,null


Group order is **not guaranteed** (groups are built in parallel). Add a `sort`, or pass `maintain_order=True` to keep first-seen order at some cost.

## 5. All four together

In [10]:
(
    orders.filter(pl.col("order_status") == "delivered")
    .with_columns(month=pl.col("order_purchase_timestamp").dt.truncate("1mo"))
    .group_by("month")
    .agg(orders=pl.len(), late_share=pl.col("is_late").mean())
    .select("month", "orders", late_pct=(pl.col("late_share") * 100).round(1))
    .sort("month")
)

month,orders,late_pct
datetime[μs],u32,f64
2016-09-01 00:00:00,1,100.0
2016-10-01 00:00:00,265,1.1
2016-12-01 00:00:00,1,0.0
2017-01-01 00:00:00,750,3.1
2017-02-01 00:00:00,1653,3.2
…,…,…
2018-04-01 00:00:00,6798,5.3
2018-05-01 00:00:00,6749,8.2
2018-06-01 00:00:00,6099,1.4


## Try it yourself

1. Which customer state has the highest late-delivery share? (Join `olist_customers_dataset.csv` on `customer_id`, then the four contexts.)
2. Rewrite section 5 with only `select` and `filter` (no `with_columns`). Which version reads better?